# Suraksha-Net: remaining training runs on Colab

**Before running:** Runtime → Change runtime type → **L4 GPU** (or A100) → Save.
Then set `PART` in the first code cell and choose **Runtime → Run all**.

Each Google account runs a different `PART`:

| Account | PART | Runs |
|---|---|---|
| Colab 1 | `"motionlite_kd"` | MotionLite + VideoMAE distillation, seeds 1–2, 5 protocols (10 runs) |
| Colab 2 | `"motionlite"` | MotionLite, seeds 1–2, 5 protocols (10 runs) |
| Colab 3 | `"mnv3_tsm"` | MobileNetV3-TSM, seeds 1–2, 5 protocols (10 runs) |

What it does: downloads RWF-2000 (Zenodo, by the dataset author, CC-BY-4.0), SFD and AIRTLab (GitHub);
checks every video against the project's SHA-256 manifest; rebuilds the frame cache; trains; saves runs
to your Google Drive (`MyDrive/suraksha_cloud/`) and finally writes `results_<PART>.zip` there.

**If Colab disconnects:** just Runtime → Run all again. Finished runs are skipped and the run in
progress resumes from its last saved epoch on Drive (the download and cache steps repeat, ~20 min).

Embedded code: commit `df9bfdce8e`, 54 files.

In [ ]:
# ===== SETTINGS: change PART for each Google account =====
PART = "motionlite_kd"      # one of: "motionlite_kd", "motionlite", "mnv3_tsm", "videomae_ft"
DRIVE_DIR = "/content/drive/MyDrive/suraksha_cloud"

In [ ]:
#@title 1. Check GPU and connect Google Drive
import os, sys, subprocess, torch
PARTS = {"motionlite_kd": "distill_teacher", "motionlite": "/motionlite/", "mnv3_tsm": "mnv3_tsm", "videomae_ft": "videomae_ft"}
assert PART in PARTS, f"PART must be one of {list(PARTS)}"
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > L4 GPU, then Run all again"
print("GPU:", torch.cuda.get_device_name(0), "| CPUs:", os.cpu_count())
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_DIR}/runs", exist_ok=True)
print("Runs will be saved to", DRIVE_DIR)

In [ ]:
#@title 2. Unpack the embedded project code (do not edit)
import base64, io, zipfile
CODE_B64 = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64))).extractall("/content/repo")
del CODE_B64
print("code commit", open("/content/repo/GIT_COMMIT").read().strip())

In [ ]:
#@title 3. Install the two missing tools (PyAV video decoder, 7-Zip)
!pip -q install av
!apt-get -qq install -y p7zip-full > /dev/null && echo "7z ready"

**4. Download the datasets** (~12 GB, about 10–20 minutes)

In [ ]:
%%bash
set -e
mkdir -p /content/dl && cd /content/dl
[ -d sfd ] || git clone -q --depth 1 https://github.com/seymanurakti/fight-detection-surv-dataset sfd
[ -d avd ] || git clone -q --depth 1 https://github.com/airtlab/A-Dataset-for-Automatic-Violence-Detection-in-Videos avd
if [ ! -d rwf ]; then
  mkdir -p rwf_parts
  seq -f "%03g" 1 13 | xargs -P 4 -I{} wget -q -c -O rwf_parts/RWF-2000.7z.{} "https://zenodo.org/records/15687512/files/RWF-2000.7z.{}?download=1"
  ls -l rwf_parts
  7z x -y -orwf rwf_parts/RWF-2000.7z.001 > /dev/null
  rm -rf rwf_parts
fi
du -sh sfd avd rwf

In [ ]:
#@title 5. Check every video against the project's SHA-256 manifest
%cd /content/repo
r = subprocess.run([sys.executable, "-m", "pipeline", "fetch", "--src", "/content/dl", "--allow-name-match"],
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "Some videos are missing: send this output to Arpit / Claude before continuing"

In [ ]:
#@title 6. Build the frame cache (~10-15 minutes)
%cd /content/repo
!python -m pipeline cache --workers {os.cpu_count()}

In [ ]:
#@title 7. Train (several hours; results go to Google Drive as each run finishes)
%cd /content/repo
W = max(2, min(8, os.cpu_count() - 2))
ONLY = PARTS[PART]
!python -u -m pipeline grid --grid pipeline/configs/grid_seeds.yaml --only "{ONLY}" --seeds 1,2 --set "paths.runs={DRIVE_DIR}/runs" --set train.workers={W} --set train.val_workers=2 2>&1 | tee -a "{DRIVE_DIR}/log_{PART}.txt"

In [ ]:
#@title 8. Package the results (small files only) -> Google Drive
import glob
out = f"{DRIVE_DIR}/results_{PART}.zip"
n = 0
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{DRIVE_DIR}/runs/**/*", recursive=True):
        if os.path.isfile(f) and not f.endswith(".pt"):
            z.write(f, os.path.relpath(f, DRIVE_DIR)); n += 1
done = len(glob.glob(f"{DRIVE_DIR}/runs/*/*/seed*/DONE"))
print(f"{out}: {n} files, {os.path.getsize(out) // 1024} KB, {done} finished runs")
print("Download this zip from Google Drive (MyDrive/suraksha_cloud) and give it to Claude on the laptop.")